In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

In [ ]:
df.head(10)
df.tail(10)
df.shape
df.columns
df.dtypes

In [ ]:
df.describe(include='all')
df.info()

In [ ]:
missing = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Percentage': (df.isnull().sum()/len(df))*100
}).sort_values('Missing Values', ascending=False)
missing
missing['Missing Values'].plot(kind='bar', figsize=(12,5))
plt.show()

In [ ]:
duplicates = df.duplicated().sum()
duplicates
df = df.drop_duplicates()

In [ ]:
unique_summary = pd.DataFrame({
    'Unique Values': df.nunique(),
    'Most Frequent': [df[c].mode().iloc[0] if not df[c].mode().empty else np.nan for c in df.columns]
})
unique_summary

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(exclude=np.number).columns

mean_df = df.copy()
median_df = df.copy()

mean_df[num_cols] = mean_df[num_cols].fillna(mean_df[num_cols].mean())
median_df[num_cols] = median_df[num_cols].fillna(median_df[num_cols].median())

for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0] if not df[c].mode().empty else 'Unknown')

In [ ]:
for col in num_cols:
    print(col)
    print(df[col].describe())
    print(df[col].mode().iloc[0])
    sns.histplot(df[col], kde=True)
    plt.show()
    sns.boxplot(x=df[col])
    plt.show()

In [ ]:
for col in cat_cols:
    print(df[col].value_counts())
    df[col].value_counts().plot(kind='bar')
    plt.show()
    df[col].value_counts().plot(kind='pie', autopct='%1.1f%%')
    plt.ylabel('')
    plt.show()

In [ ]:
iqr_results = []
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5*iqr
    upper = q3 + 1.5*iqr
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    iqr_results.append([col,q1,q3,iqr,lower,upper,outliers])
iqr_df = pd.DataFrame(iqr_results, columns=['Column','Q1','Q3','IQR','Lower','Upper','Outliers'])
iqr_df

In [ ]:
z_outliers = {}
for col in num_cols:
    z = np.abs(zscore(df[col]))
    z_outliers[col] = (z > 3).sum()
pd.DataFrame(z_outliers.items(), columns=['Column','Outliers'])

In [ ]:
capped_df = df.copy()
for col in num_cols:
    q1 = capped_df[col].quantile(0.25)
    q3 = capped_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5*iqr
    upper = q3 + 1.5*iqr
    capped_df[col] = capped_df[col].clip(lower, upper)

In [ ]:
df['AgeGroup'] = pd.cut(df['Age'], bins=[18,25,35,45,100], labels=['Young','Mid','Senior','Veteran'])
df['IncomeCategory'] = pd.qcut(df['MonthlyIncome'], 3, labels=['Low','Medium','High'])
df['ExperienceCategory'] = pd.cut(df['TotalWorkingYears'], bins=[0,5,10,20,50], labels=['Beginner','Intermediate','Experienced','Expert'])
df['PromotionRiskScore'] = df['YearsSinceLastPromotion'] * df['JobLevel']
df['AttritionRiskScore'] = (df['DistanceFromHome'] + df['MonthlyIncome'].rank(pct=True) + (df['OverTime']=='Yes').astype(int))

In [ ]:
corr = df.select_dtypes(include=np.number).corr()
plt.figure(figsize=(14,10))
sns.heatmap(corr, cmap='coolwarm')
plt.show()

In [ ]:
for col in cat_cols:
    if col != 'Attrition':
        sns.boxplot(x=df[col], y=df['MonthlyIncome'])
        plt.xticks(rotation=90)
        plt.show()

In [ ]:
pd.crosstab(df['Attrition'], df['OverTime'])
sns.heatmap(pd.crosstab(df['Attrition'], df['OverTime']), annot=True)
plt.show()

In [ ]:
standard = pd.DataFrame(StandardScaler().fit_transform(df[num_cols]), columns=num_cols)
minmax = pd.DataFrame(MinMaxScaler().fit_transform(df[num_cols]), columns=num_cols)
robust = pd.DataFrame(RobustScaler().fit_transform(df[num_cols]), columns=num_cols)

standard.describe()
minmax.describe()
robust.describe()

In [ ]:
skew_before = df[num_cols].skew()
log_df = np.log1p(df[num_cols])
sqrt_df = np.sqrt(df[num_cols])
skew_after_log = log_df.skew()
skew_after_sqrt = sqrt_df.skew()

pd.DataFrame({
    'Before': skew_before,
    'Log': skew_after_log,
    'Sqrt': skew_after_sqrt
})

In [ ]:
checks = {
    'Null MonthlyIncome': df['MonthlyIncome'].isnull().sum(),
    'Negative Age': (df['Age'] < 0).sum(),
    'Negative YearsAtCompany': (df['YearsAtCompany'] < 0).sum(),
    'YearsAtCompany > TotalWorkingYears': (df['YearsAtCompany'] > df['TotalWorkingYears']).sum(),
    'MonthlyIncome <= 0': (df['MonthlyIncome'] <= 0).sum(),
    'Age <= 18': (df['Age'] <= 18).sum(),
    'Duplicate EmployeeNumber': df['EmployeeNumber'].duplicated().sum()
}
checks

In [ ]:
drop_cols = ['EmployeeCount','Over18','StandardHours']
encode_cols = list(df.select_dtypes(exclude=np.number).columns)
scale_cols = list(df.select_dtypes(include=np.number).columns)

print(drop_cols)
print(encode_cols)
print(scale_cols)

In [ ]:
df.to_csv('cleaned_hr_dataset.csv', index=False)